*First attempt*

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')
import re
import nltk
from nltk.corpus import stopwords
import string
from nltk.stem.isri import ISRIStemmer


In [ ]:
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('averaged_perceptron_tagger')
arabic_stopwords = set(stopwords.words('arabic'))

In [ ]:
!kaggle datasets download -d alizahidraja/quran-nlp
!unzip /content/quran-nlp.zip
!pip install -U farasapy
!pip install fasttext
!pip install snowballstemmer
!pip install deep-translator
!git clone https://github.com/abdelrahmaan/Hadith-Data-Sets.git


In [ ]:
data_frame=pd.read_csv("/content/data/hadith/kaggle_hadiths_clean.csv")

In [ ]:
data_frame.info()

In [ ]:
nan_rows = data_frame[data_frame['text_en'].isna()]


In [ ]:
text=nan_rows['text_ar'].values[100]
text

In [ ]:
xx=text.replace("\u200f",'').strip()
clean_text = re.sub(r'\s+', ' ', xx).strip()
clean_text

In [ ]:
df=pd.read_csv("/content/data/hadith/Sanadset 650K Data on Hadith Narrators/sanadset.csv")

In [ ]:
# h_books=df["Book"].unique()
mask=df["Book"]=="صحيح البخاري"
df_Bukhari=df[mask]

In [ ]:
df_Bukhari.info()

In [ ]:
from farasa.stemmer import FarasaStemmer
# text="متعة"
# stemmer = FarasaStemmer()
# res1=stemmer.stem(text)
# print(res1)


In [ ]:
df_Bukhari["hadith_only_col"]=df_Bukhari['Matn'].apply(hadith_only)
df_Bukhari['cleaned_text'] = df_Bukhari['hadith_only_col'].apply(preprocess_text)


In [ ]:
for i,w in enumerate(words_dict):
  word_to_ix[w]=i
  ix_to_word[i]=w


In [ ]:
import gensim
from gensim.models import Word2Vec
from gensim.models import KeyedVectors
import gensim.downloader as api
# !wget "https://bakrianoo.ewr1.vultrobjects.com/aravec/full_grams_cbow_100_twitter.zip"
# !unzip "full_grams_cbow_100_twitter.zip"
model2=gensim.models.Word2Vec.load("full_grams_cbow_100_twitter.mdl")


In [ ]:
print(len(words_dict)-14577)

In [ ]:
# if "الحث" in model.wv.index_to_key:
#   print("yes")
# else:
#   print("no")

model2.wv.most_similar("الحث")

In [ ]:
import fasttext.util
fasttext.util.download_model('ar', if_exists='ignore')

In [ ]:
!gunzip  /content/cc.ar.300.bin.gz

In [ ]:
import fasttext
model = fasttext.load_model('cc.ar.300.bin')

In [ ]:
c=0
for i in words_dict:
  if i not in model.words:
    c+=1
print(c) #3735


In [ ]:
# # dir(model)
# model.get_analogies("القاهرة", "مصر", "امريكا")
# # words, labels = model.get_line("مثال عملي للتطور")
# model.get_nearest_neighbors("مصر")
# subwords, ids = model.get_subwords("playing")
# print(subwords)
# print(ids)


In [ ]:
h_vec=[]
c=0
for i in df_Bukhari['cleaned_text'].values:
  if i=="":
    c+=1
    continue
  h_vec.append((c,model.get_sentence_vector(i)))
  c+=1



In [ ]:
x="جاء الرسول عليه الصلاة والسلام من جهة بئر جمل، فقابله شخص وألقى عليه التحية،ولكنه لم يجبه"
x=preprocess_text(x)
x2=df_Bukhari.iloc[3200]['cleaned_text']
cluc_cos_similarity(model.get_sentence_vector(x),model.get_sentence_vector(x2))


In [ ]:
def get_nearst_hadith(hadith,h_vectors):
  hadith=preprocess_text(hadith)
  score_val=[]
  sent_vec=model.get_sentence_vector(hadith)
  for h_num,sent_vec2 in h_vectors:
    value=cluc_cos_similarity(sent_vec,sent_vec2)
    score_val.append((h_num,value))
  h_nums=[i[0] for i in score_val]
  score_val2=[i[1] for i in score_val]
  score_val2=np.array(score_val2)
  indices = np.argsort(score_val2)[-5:]
  largest_three = indices[::-1]
  largest_three2=[]
  for i in largest_three:
    largest_three2.append(h_nums[i])
  # largest_three2=largest_three2[::-1]
  # largest_three=largest_three[::-1]
  hadiths=[]
  for i,i2 in zip(largest_three2,largest_three):
    x=df_Bukhari.iloc[i]['Hadith']
    clean_text = re.sub(r'<[^>]+>', '', x)
    clean_text = re.sub(r'\s+', ' ', clean_text).strip()
    hadiths.append((clean_text,score_val2[i2]))
  return hadiths

# test2="عن مَعقلِ بنِ يسارٍ قال: كانت أختُه تحتَ رجلٍ فطلَّقها ثمَّ خلَّى عنها حتَّى انقَضَت عدَّتُها ثمَّ قرَّب يخطُبُها فحمِي معقلٌ مِن ذلك وقال: خلَّى عنها وهو يقدِرُ عليها فحال بينَه وبينَها فأنزَل اللهُ: { وَإِذَا طَلَّقْتُمُ النِّسَاءَ فَبَلَغْنَ أَجَلَهُنَّ فَلَا تَعْضُلُوهُنَّ أَنْ يَنْكِحْنَ أَزْوَاجَهُنَّ إِذَا تَرَاضَوْا بَيْنَهُمْ بِالْمَعْرُوف } (البقرة: 232) »"
x="جاء الرسول عليه الصلاة والسلام من جهة بئر جمل، فقابله شخص وألقى عليه التحية،ولكنه لم يجبه"
# test2="جاء الرسول عليه الصلاة والسلام من جهة بئر جمل، فقابله شخص وألقى عليه التحية،ولكنه لم يجبه فمسح بوجهه ويديه من ثم قام برد السلام"
get_nearst_hadith(x,h_vec)

In [ ]:
def words_counter(sent,words):
    res=np.zeros(shape=(len(sent),(len(words))))
    for i in range(len(sent)):
        for j in range(len(words)):
            if words[j] in sent[i]:
                res[i,j]=1
    return res

def get_count_vec(sent_1,sent_2):
  arabic_sent=[sent_1,sent_2]
  full_sent=""
  words=[]
  new_words=[]
  for i in arabic_sent:
    full_sent+=i
    full_sent+=" "
    words=nltk.word_tokenize(full_sent)
  for j in words:
    new_words.append(j)
  new_words=list(set(new_words))
  return words_counter(arabic_sent,new_words)

test=df_Bukhari['cleaned_text'].iloc[50]
test=preprocess_text(test)
test2="قال رسول الله صلى الله عليه وسلم من عمل عملا ليس عليه أمرنا فهو رد"

# get_count_vec(test,test2)
# score=cluc_cos_similarity(get_count_vec(test,test2)[0],get_count_vec(test,test2)[1])

# score

# def get_nearst_hadith2(hadith):
#   hadith=preprocess_text(hadith)
#   words=nltk.word_tokenize(hadith)
#   x=0
#   for i in words:
#     if i not in words_dict:
#       words[x]=get_nearst_word_in_dict(i,words_dict)
#     x+=1
#   hadith=" ".join(words)
#   hadith=hadith.strip()
#   score_val=[]
#   for h in df_Bukhari['cleaned_text']:
#     x=get_count_vec(hadith,h)
#     value=cluc_cos_similarity(x[0],x[1])
#     if str(value)=="nan":
#       value=0
#     score_val.append(value)
#   indices = np.argsort(score_val)[-3:]
#   largest_three = indices[::-1]
#   hadiths=[]
#   for i in largest_three:
#     x=df_Bukhari.iloc[i]['Hadith']
#     clean_text = re.sub(r'<[^>]+>', '', x)
#     clean_text = re.sub(r'\s+', ' ', clean_text).strip()
#     hadiths.append((clean_text,score_val[i]))
#   return hadiths

# txt="أتيت إلى الرسول عليه الصلاة والسلام ومعي أخ لي ليضع التمر في فمه ليعوذه، بينما هو في حظيرته، فرأيته يضع علامة على نعجة، وظننته قال في طرف أذنها."
# # words=nltk.word_tokenize(txt)
# # for i in words:
# #   if i in words_dict:
# #     print((i,"yes"))
# #   else:
# #     # print(get_nearst_word_in_dict(i,words_dict))
# #     print((i,"no"))

# get_nearst_hadith2(txt)

In [ ]:
from transformers import AutoModel
from scipy.spatial.distance import cosine
import sklearn.metrics.pairwise

model = AutoModel.from_pretrained("jinaai/jina-embeddings-v3", trust_remote_code=True)

def embed_text(input):
  embeddings = model.encode(input, task="text-matching")
  return embeddings



In [ ]:
emded_array=[]
start_time = time.time()
x=0
for i in h_list[:100]:
  res=embed_text(i)
  emded_array.append(res)
  if x==50:
    print("--- %s minute ---" % ((time.time() - start_time)//60))
  x+=1


In [ ]:
# !wget https://huggingface.co/azizalto/arabic-news-embeddings/resolve/main/arabic-news.tar.gz
# !tar -xvzf arabic-news.tar.gz



In [ ]:
!git clone https://github.com/abdelrahmaan/Hadith-Data-Sets.git


In [ ]:
!pip3 install tensorflow_text
!pip3 install --upgrade tensorflow-gpu
!pip install tensorflow-hub
!pip install bokeh
!pip install simpleneighbors
!pip install tqdm
!pip install nltk


In [ ]:
import bokeh
import bokeh.models
import bokeh.plotting
import numpy as np
import os
import pandas as pd
import tensorflow.compat.v2 as tf
import tensorflow_hub as hub
from tensorflow_text import SentencepieceTokenizer
import sklearn.metrics.pairwise
import nltk
nltk.download('punkt')
from simpleneighbors import SimpleNeighbors
from tqdm import tqdm
from tqdm import trange


In [ ]:
module_url = 'https://tfhub.dev/google/universal-sentence-encoder-multilingual/3' #@param ['https://tfhub.dev/google/universal-sentence-encoder-multilingual/3', 'https://tfhub.dev/google/universal-sentence-encoder-multilingual-large/3']

model = hub.load(module_url)

def embed_text(input):
  return model(input)

In [ ]:
def cos_sim_data(data_feature_vectors,query_feature_vector):
    similarities_scores = sklearn.metrics.pairwise.cosine_similarity(data_feature_vectors,query_feature_vector)
    return similarities_scores

In [ ]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer

df = pd.read_csv("/content/Hadith-Data-Sets/All Hadith Books/Sahih Bukhari Without_Tashkel.csv")
df.head()

In [ ]:
import json

# this functions make a mgrams from 2 : 5 gram by Tf-idf for each row 'hadith'
# make a dict for save index for each hadith and all compination of gram's
# return all grams for each hadith as a dict {index: grams}
# make a json file to save dict and to saving time

def make_grams():
  all_grams_clusters = {}
  vectorizer_1_2_3_gram = TfidfVectorizer(analyzer='word', ngram_range=(2,5))
  for i in range(df.shape[0]): # df.shape[0]
    grams_for_each_row = []
    vectors_1_2_3_gram = vectorizer_1_2_3_gram.fit_transform([df['Sahih Bukhari Without_Tashkel'][i]])
    feature_names_1_2_3_gram = vectorizer_1_2_3_gram.get_feature_names_out()
    grams_for_each_row.append(feature_names_1_2_3_gram)
    all_grams_clusters[str(i)] = grams_for_each_row

    # # save all similarities with indeces as json
    # with open('similarities.txt', 'w') as json_file:
    #   json.dump(all_grams_clusters, json_file)

  # print(all_grams_clusters)

  return all_grams_clusters

In [ ]:
def top_n_similarities(query,loaded_array,n = 3):
  all_sim_clusters = {}
  top_n_similarities_list = []
  for hadith_num in range(len(grams_clusters)):
    similarities_array = cos_sim_data(np.array(embed_text(query)),loaded_array[hadith_num])
    all_sim_clusters[str(hadith_num)] = np.max(np.array(similarities_array).tolist())
    all_sim_clusters = {k: v for k, v in sorted(all_sim_clusters.items(), key=lambda item: item[1] , reverse= True)}
  for index, element in enumerate(all_sim_clusters.items()):
    if index == n:
      break
    top_n_similarities_list.append(element)
  return top_n_similarities_list


In [ ]:
import time
import pickle

grams_clusters = make_grams()



In [ ]:
emded_array=[]
start_time = time.time()
for hadith_num, grams_list in grams_clusters.items():
  res=embed_text(grams_list[0])
  emded_array.append(res)
  if hadith_num==str(100):
    print("--- %s minute ---" % ((time.time() - start_time)//60))

# with open('emded_array.pkl', 'wb') as file:
#     pickle.dump(emded_array, file)


# with open('emded_array.pkl', 'rb') as file:
#     loaded_array = pickle.load(file)

In [ ]:
df.iloc[50].values

In [ ]:
query = "من علامات القيامة الجهل "
list_of_similarities_scores = top_n_similarities(query,emded_array,3)
for index, score in list_of_similarities_scores:
  print('Hadith number in Bukhari: {}\nHadiths is:\n{}\nScore: {}\n{}'.format(int(index), df['Sahih Bukhari Without_Tashkel'][int(index)], score, '-'*200))


In [ ]:
def top_n_similarities2(query,loaded_array,n = 3):
  similarities_list = []
  for hadith_num in range(len(loaded_array)):
    similarities_array = 1-(cosine(np.array(embed_text(query)),loaded_array[hadith_num]))
    similarities_list.append(similarities_array)
  indices = np.argsort(similarities_list)[(-n):]
  largest_three = indices[::-1]
  hadiths=[]
  for i in largest_three:
    x=df.iloc[i]
    hadiths.append(x.values)
  return hadiths, largest_three



In [ ]:
query = "من علامات القيامة الجهل "#"شتم المؤمن فسق و الاقتتال معه يعد كفر وحيود عن الايمان او كما قال النبي"
H,list_of_similarities_scores = top_n_similarities2(query,emded_array,3)
# for index, score in list_of_similarities_scores:
#   print('Hadith number in Bukhari: {}\nHadiths is:\n{}\nScore: {}\n{}'.format(int(index), df['Sahih Bukhari Without_Tashkel'][int(index)], score, '-'*200))


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')
import re
import nltk
from nltk.corpus import stopwords
import string
import string
from nltk.stem import WordNetLemmatizer

In [ ]:
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('averaged_perceptron_tagger')
arabic_stopwords = set(stopwords.words('arabic'))
english_stopwords = set(stopwords.words('english'))

In [ ]:
!kaggle datasets download -d alizahidraja/quran-nlp
!unzip /content/quran-nlp.zip
!pip install deep-translator
!pip install fasttext
# !pip install sentence_transformers
# !pip install fasttext
# !git clone https://github.com/abdelrahmaan/Hadith-Data-Sets.git


In [ ]:
data_frame=pd.read_csv("/content/data/hadith/kaggle_hadiths_clean.csv")

In [ ]:
data_frame.info()

In [ ]:
# import spacy
# nlp = spacy.load('en_core_web_sm')
def preprocess_text(text):
  if text=="" or text=="Nan" or pd.isna(text):
    return ""
  text = re.sub(r"[``“’”‘ﷺ]", '', text).strip()
  txt=text.replace("\u200f",'').strip()
  clean_text = re.sub(r'\s+', ' ', txt).strip()
  clean_text=clean_text.lower()
  words=nltk.word_tokenize(clean_text)
  filtered_words = [word for word in words if word not in string.punctuation]
  filtered_txt=" ".join(filtered_words)
  cleaned_text = re.sub(r"""[`"']""", '', filtered_txt)
  # lemmatized_sentence = " ".join([token.lemma_ for token in nlp(filtered_txt)])
  cleaned_text = re.sub(r'\s+', ' ', cleaned_text)
  return cleaned_text

In [ ]:
data_frame["cleaned_text_en"]=data_frame["text_en"].apply(preprocess_text)

In [ ]:
data_frame.head()

In [ ]:
from deep_translator import GoogleTranslator
def translate_text(text):
  translated = GoogleTranslator(source='ar', target='en').translate(text)
  return preprocess_text(translated)

def translate_text_ar(text):
  translated = GoogleTranslator(source='en', target='ar').translate(text)
  return translated

In [ ]:
null_val=data_frame[data_frame['text_en'].isna()]


In [ ]:
null_val=data_frame[data_frame['text_en'].isna()]
trans_txt=[]
ind=null_val.index.values
for i,h_index in zip(null_val["text_ar"].values,ind):
  trans_txt.append((h_index,translate_text(i)))



In [ ]:
for i in trans_txt:
  data_frame.at[i[0], "cleaned_text_en"] = i[1]



In [ ]:
df=pd.read_csv("/content/drive/MyDrive/hadith_project_folder/final_cleaned_data.csv")

In [ ]:
# df.info()
# df.drop("sentences_embedd",inplace=True,axis=1)

In [ ]:
!pip3 install tensorflow_text
!pip3 install --upgrade tensorflow-gpu
!pip install tensorflow-hub
!pip install bokeh
!pip install simpleneighbors
!pip install tqdm
!pip install nltk


In [ ]:
import bokeh
import bokeh.models
import bokeh.plotting
import numpy as np
import os
import pandas as pd
import tensorflow.compat.v2 as tf
import tensorflow_hub as hub
from tensorflow_text import SentencepieceTokenizer
import sklearn.metrics.pairwise
import nltk
nltk.download('punkt')
from simpleneighbors import SimpleNeighbors
from tqdm import tqdm
from tqdm import trange
from sklearn.feature_extraction.text import TfidfVectorizer

In [ ]:
module_url = 'https://tfhub.dev/google/universal-sentence-encoder-multilingual/3' #@param ['https://tfhub.dev/google/universal-sentence-encoder-multilingual/3', 'https://tfhub.dev/google/universal-sentence-encoder-multilingual-large/3']

model = hub.load(module_url)

def embed_text(input):
  return model(input)

In [ ]:
def cos_sim_data(data_feature_vectors,query_feature_vector):
    similarities_scores = sklearn.metrics.pairwise.cosine_similarity(data_feature_vectors,query_feature_vector)
    return similarities_scores

In [ ]:
import json

def make_grams():
  all_grams_clusters = {}
  vectorizer_1_2_3_gram = TfidfVectorizer(analyzer='word', ngram_range=(2,5))
  for i in range(df.shape[0]): # df.shape[0]
    grams_for_each_row = []
    vectors_1_2_3_gram = vectorizer_1_2_3_gram.fit_transform([df['cleaned_text_en'][i]])
    feature_names_1_2_3_gram = vectorizer_1_2_3_gram.get_feature_names_out()
    grams_for_each_row.append(feature_names_1_2_3_gram)
    all_grams_clusters[str(i)] = grams_for_each_row

    # # save all similarities with indeces as json
    # with open('similarities.txt', 'w') as json_file:
    #   json.dump(all_grams_clusters, json_file)

  # print(all_grams_clusters)

  return all_grams_clusters

In [ ]:
import json

def make_grams():
  all_grams_clusters = {}
  vectorizer_1_2_3_gram = TfidfVectorizer(analyzer='word', ngram_range=(1,5))  # Changed ngram_range to (1,5)
  for i in range(df.shape[0]): # df.shape[0]
    grams_for_each_row = []

    # Check if the text is empty after preprocessing
    text = df['cleaned_text_en'][i]
    if text.strip(): # proceed only if the text is not empty after stripping whitespace
        vectors_1_2_3_gram = vectorizer_1_2_3_gram.fit_transform([text])
        feature_names_1_2_3_gram = vectorizer_1_2_3_gram.get_feature_names_out()
        grams_for_each_row.append(feature_names_1_2_3_gram)
        all_grams_clusters[str(i)] = grams_for_each_row
    else:
        print(f"Warning: Empty text at index {i}. Skipping...") # print warning message

  return all_grams_clusters

In [ ]:
import time
import pickle

grams_clusters = make_grams()



In [ ]:
emded_array=[]
start_time = time.time()
for hadith_num, grams_list in grams_clusters.items():
  res=embed_text(grams_list[0])
  emded_array.append(res)
  if hadith_num==str(1000):
    print("--- %s minute ---" % ((time.time() - start_time)//60))

# with open('emded_array.pkl', 'wb') as file:
#     pickle.dump(emded_array, file)


# with open('emded_array.pkl', 'rb') as file:
#     loaded_array = pickle.load(file)

In [ ]:
len(emded_array)

In [ ]:
def top_n_similarities(query,loaded_array,n = 3):
  all_sim_clusters = {}
  top_n_similarities_list = []
  for hadith_num in range(3000):
    similarities_array = cos_sim_data(np.array(embed_text(query)),loaded_array[hadith_num])
    all_sim_clusters[str(hadith_num)] = np.max(np.array(similarities_array).tolist())
    all_sim_clusters = {k: v for k, v in sorted(all_sim_clusters.items(), key=lambda item: item[1] , reverse= True)}
  for index, element in enumerate(all_sim_clusters.items()):
    if index == n:
      break
    top_n_similarities_list.append(element)
  return top_n_similarities_list


In [ ]:
# test2="عن مَعقلِ بنِ يسارٍ قال: كانت أختُه تحتَ رجلٍ فطلَّقها ثمَّ خلَّى عنها حتَّى انقَضَت عدَّتُها ثمَّ قرَّب يخطُبُها فحمِي معقلٌ مِن ذلك وقال: خلَّى عنها وهو يقدِرُ عليها فحال بينَه وبينَها فأنزَل اللهُ: { وَإِذَا طَلَّقْتُمُ النِّسَاءَ فَبَلَغْنَ أَجَلَهُنَّ فَلَا تَعْضُلُوهُنَّ أَنْ يَنْكِحْنَ أَزْوَاجَهُنَّ إِذَا تَرَاضَوْا بَيْنَهُمْ بِالْمَعْرُوف } (البقرة: 232) »"
# test2="جاء الرسول عليه الصلاة والسلام من جهة بئر جمل، فقابله شخص وألقى عليه التحية،ولكنه لم يجبه فمسح بوجهه ويديه من ثم قام برد السلام"

# test2="نهى النبي عن المتعة في غزوة خيبر"
test2="استحلال المعازف في اواخر الزمن"
txt=translate_text(test2)
top_n_similarities(txt,emded_array)

In [ ]:
df.iloc[929]['text_ar']

In [ ]:
# data_frame[data_frame['text_en'].isna()]


In [ ]:
data_frame.to_csv("final_cleaned_data.csv")


In [ ]:
df=pd.read_csv("/content/drive/MyDrive/hadith_project_folder/final_cleaned_data.csv")

In [ ]:
df.info()

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

def get_nearst_hadith(hadith,h_vectors,n=3):
  hadith=translate_text(hadith)
  score_val=[]
  sent_vec=embed_text(hadith).numpy().reshape(1,-1)
  for sent_vec2 in h_vectors:
    sent_vec2=sent_vec2.numpy().reshape(1,-1)
    value=cosine_similarity(sent_vec,sent_vec2)
    score_val.append(value[0][0])
  score_val2=np.array(score_val)
  indices = np.argsort(score_val2)[-n:]
  largest_three = indices[::-1]
  scores=score_val2[[largest_three]]
  return largest_three,scores

In [ ]:
from transformers import AutoModel

model = AutoModel.from_pretrained("jinaai/jina-embeddings-v3", trust_remote_code=True)
def embed_text(input):
  embeddings = model.encode(input, task="text-matching")
  return embeddings

In [ ]:
h_vec_=[]
for x in df['cleaned_text_en'].iloc[:]:
  h_vec_.append(embed_text(x))




In [ ]:
h_vec_=[]
for x in df['sentences_embedd']:
  x = x.replace('\n', ' ').replace('[', '').replace(']', '')
  h_vec_.append(np.array(x.split(), dtype=np.float64))

In [ ]:
# test2="عن مَعقلِ بنِ يسارٍ قال: كانت أختُه تحتَ رجلٍ فطلَّقها ثمَّ خلَّى عنها حتَّى انقَضَت عدَّتُها ثمَّ قرَّب يخطُبُها فحمِي معقلٌ مِن ذلك وقال: خلَّى عنها وهو يقدِرُ عليها فحال بينَه وبينَها فأنزَل اللهُ: { وَإِذَا طَلَّقْتُمُ النِّسَاءَ فَبَلَغْنَ أَجَلَهُنَّ فَلَا تَعْضُلُوهُنَّ أَنْ يَنْكِحْنَ أَزْوَاجَهُنَّ إِذَا تَرَاضَوْا بَيْنَهُمْ بِالْمَعْرُوف } (البقرة: 232) »"
# test2="جاء الرسول عليه الصلاة والسلام من جهة بئر جمل، فقابله شخص وألقى عليه التحية،ولكنه لم يجبه فمسح بوجهه ويديه من ثم قام برد السلام"

# test2="نهى النبي عن المتعة في غزوة خيبر"
test2="استحلال المعازف في اواخر الزمن"

In [ ]:
res=get_nearst_hadith(test2,h_vec_,5)

In [ ]:
for index, score in zip(res[0],res[1][0]):
  # print('Hadith number : {}\nHadiths is:\n{}\nScore: {}\n{}'.format(int(index),df.iloc[index]['text_ar'], score, '-'*200))
  arabic_text = re.findall(r'[\u0600-\u06FF]+', df.iloc[index]['chapter'])
  arabic_string = ' '.join(arabic_text)
  print(f"الحديث رقم : {df.iloc[index]['hadith_no']} \n\n الحديث هو : {df.iloc[index]['text_ar']} \n\n كتاب : {translate_text_ar(df.iloc[index]['source'])} \n\n باب : {arabic_string} \n\n نسبة التشابه : {score} ")
  print("-"*100)


In [ ]:
!pip install fastapi nest-asyncio pyngrok uvicorn
!pip install "uvicorn[standard]"
!pip install python-multipart
# Configure NGROK_AUTHTOKEN through your environment; never hard-code it.

In [ ]:
from fastapi import FastAPI, UploadFile, File
from fastapi.responses import StreamingResponse,JSONResponse,FileResponse

In [ ]:
app = FastAPI()

@app.get("/get_hadith")
async def get_nearst_hadith2(text:str):
  res=get_nearst_hadith(text,h_vec_)
  final_res=[]
  for index, score in zip(res[0],res[1][0]):
    arabic_text = re.findall(r'[\u0600-\u06FF]+', df.iloc[index]['chapter'])
    arabic_string = ' '.join(arabic_text)
    final_res.append({f"رقم الحديث": {df.iloc[index]['hadith_no']} ,"نص الحديث"  : {df.iloc[index]['text_ar']} ,"كتاب" : {translate_text_ar(df.iloc[index]['source'])}, "باب" : {arabic_string},"نسبة التشابه ": {score}})
  return final_res



In [ ]:
# for x in df['source'].unique():
#   print(translate_text_ar(x))

In [ ]:
import nest_asyncio
from pyngrok import ngrok
import uvicorn

ngrok_tunnel = ngrok.connect(8000)
print('Public URL:', ngrok_tunnel.public_url)
nest_asyncio.apply()
uvicorn.run(app, port=8000)